In [ ]:
from pathlib import Path
import re
from pprint import pprint

# 中文註解：設定要比較的兩份 ontology DOT 檔案路徑。
OLD_ONTOLOGY_PATH = Path("ontology_20.dot")
NEW_ONTOLOGY_PATH = Path("ontology_20_new.dot")

# 中文註解：這個正規表示式會抓出 DOT 裡每一條 relation edge 的 source、target 與 label。
EDGE_PATTERN = re.compile(
    r'"(?P<source>[^"]+)"\s*->\s*"(?P<target>[^"]+)"\s*\[label="(?P<label>[^"]+)"\]'
)


def parse_dot_ontology(path):
    """中文註解：讀取 DOT ontology，回傳節點集合、完整 relation 集合，以及用來比對 parent-child 關係的索引。"""
    text = path.read_text(encoding="utf-8-sig")
    edges = set()
    nodes = set()
    relation_by_pair = {}

    # 中文註解：逐行解析 relation，並把 source / target 都加入 class node 清單。
    for match in EDGE_PATTERN.finditer(text):
        source = match.group("source").strip()
        target = match.group("target").strip()
        label = match.group("label").strip()
        edge = (source, target, label)

        nodes.update([source, target])
        edges.add(edge)
        relation_by_pair[(source, target)] = label

    return {
        "path": path,
        "nodes": nodes,
        "edges": edges,
        "relation_by_pair": relation_by_pair,
    }


def compare_ontologies(old_path, new_path):
    """中文註解：比較新舊 ontology，整理成容易閱讀與後續程式處理的差異 list。"""
    old_ontology = parse_dot_ontology(old_path)
    new_ontology = parse_dot_ontology(new_path)

    old_nodes = old_ontology["nodes"]
    new_nodes = new_ontology["nodes"]
    old_edges = old_ontology["edges"]
    new_edges = new_ontology["edges"]
    old_relation_by_pair = old_ontology["relation_by_pair"]
    new_relation_by_pair = new_ontology["relation_by_pair"]

    difference_list = []

    # 中文註解：列出新版 ontology 新增或移除的 class node。
    for node in sorted(new_nodes - old_nodes, key=str.lower):
        difference_list.append({"type": "added_node", "node": node})

    for node in sorted(old_nodes - new_nodes, key=str.lower):
        difference_list.append({"type": "removed_node", "node": node})

    # 中文註解：如果 source -> target 還在，但 relation label 改變，單獨列為 relation_changed。
    old_pairs = set(old_relation_by_pair)
    new_pairs = set(new_relation_by_pair)
    for pair in sorted(old_pairs & new_pairs, key=lambda item: (item[0].lower(), item[1].lower())):
        old_label = old_relation_by_pair[pair]
        new_label = new_relation_by_pair[pair]
        if old_label != new_label:
            difference_list.append({
                "type": "relation_changed",
                "source": pair[0],
                "target": pair[1],
                "old_relation": old_label,
                "new_relation": new_label,
            })

    # 中文註解：列出新版 ontology 新增或移除的完整 edge，包含 relation label。
    for source, target, label in sorted(new_edges - old_edges, key=lambda item: (item[0].lower(), item[1].lower(), item[2].lower())):
        if (source, target) not in old_relation_by_pair:
            difference_list.append({
                "type": "added_edge",
                "source": source,
                "target": target,
                "relation": label,
            })

    for source, target, label in sorted(old_edges - new_edges, key=lambda item: (item[0].lower(), item[1].lower(), item[2].lower())):
        if (source, target) not in new_relation_by_pair:
            difference_list.append({
                "type": "removed_edge",
                "source": source,
                "target": target,
                "relation": label,
            })

    return difference_list


# 中文註解：執行比較並印出差異 list；difference_list 也會保留在 notebook 變數中方便後續分析。
difference_list = compare_ontologies(OLD_ONTOLOGY_PATH, NEW_ONTOLOGY_PATH)

print(f"Total differences: {len(difference_list)}")
pprint(difference_list, sort_dicts=False)


Total differences: 26
[{'type': 'removed_node', 'node': 'Adaptive Reuse'},
 {'type': 'removed_node', 'node': 'curtain wall'},
 {'type': 'removed_node', 'node': 'daylighting'},
 {'type': 'removed_node', 'node': 'elevator'},
 {'type': 'removed_node', 'node': 'green roof'},
 {'type': 'removed_node', 'node': 'Green Space'},
 {'type': 'removed_node', 'node': 'local government'},
 {'type': 'removed_node', 'node': 'mass timber'},
 {'type': 'removed_node', 'node': 'modularity'},
 {'type': 'removed_node', 'node': 'prefabrication'},
 {'type': 'removed_node', 'node': 'public transport'},
 {'type': 'removed_node', 'node': 'Public Transport Node'},
 {'type': 'removed_node', 'node': 'view corridor'},
 {'type': 'removed_edge',
  'source': 'construction method',
  'target': 'prefabrication',
  'relation': 'sense'},
 {'type': 'removed_edge',
  'source': 'design concept',
  'target': 'modularity',
  'relation': 'sense'},
 {'type': 'removed_edge',
  'source': 'facade system',
  'target': 'curtain wall',
